# 🔗 Joins y concat: bicicletas públicas de Concepción

**Taller de Obtención y Preparación de Datos** · batería de práctica del [Visualizador TOPD](https://mati3939.github.io/visualizador-numpy-pandas/)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mati3939/visualizador-numpy-pandas/blob/main/notebooks/09_joins.ipynb)

**Objetivos:** combinar tablas con `merge` (inner/left, llaves con otro nombre, doble join con sufijos) y apilar periodos con `concat` sin romper el índice.

🔎 **Apóyate en el visualizador:** [Joins y concat](https://mati3939.github.io/visualizador-numpy-pandas/#merge) — ten la pestaña abierta mientras resuelves; cada concepto de aquí está animado allá.

---

## Contexto: el sistema de ciclovías penquista

La municipalidad opera estaciones de bicicletas públicas en el Gran Concepción. Los viajes se registran con el **id** de la estación, pero los reportes necesitan **nombres** — el escenario exacto de la clase 20 del curso. Datos chicos y a mano, para que puedas verificar cada join a ojo.

## 0 · Preparación

In [ ]:
import pandas as pd

estaciones = pd.DataFrame({
    'id_estacion': [1, 2, 3, 4, 5, 6],
    'nombre': ['Plaza Perú', 'Parque Ecuador', 'Estación Central',
               'Laguna Redonda', 'Costanera', 'Universidad'],
    'comuna': ['Concepción', 'Concepción', 'Concepción',
               'Concepción', 'Hualpén', 'Concepción'],
})
viajes = pd.DataFrame({
    'id_viaje': range(1, 13),
    'salida': [1, 2, 1, 3, 5, 2, 1, 4, 2, 9, 3, 1],   # ojo con el 9…
    'llegada': [2, 1, 3, 5, 1, 4, 2, 1, 3, 2, 1, 5],
    'duracion_min': [12, 15, 8, 22, 31, 9, 14, 18, 11, 25, 16, 7],
})
print(estaciones)
print(viajes)

## 1 · Calentamiento

Ten abierto [https://mati3939.github.io/visualizador-numpy-pandas/#merge](https://mati3939.github.io/visualizador-numpy-pandas/#merge): las líneas entre llaves que vas a imaginar aquí, allá están dibujadas.

### 1.1 El primer merge (y una fila que desaparece) ⭐

Une `viajes` con `estaciones` por la estación de **salida** usando `pd.merge(..., left_on='salida', right_on='id_estacion', how='inner')` en `con_nombre`. Compara `len(viajes)` con `len(con_nombre)`: ¿cuántos viajes se perdieron y por qué?

<details><summary>💡 Pista (haz clic)</summary>

Las llaves se llaman distinto en cada tabla (`salida` vs `id_estacion`): para eso existen `left_on` y `right_on`.

</details>

In [ ]:
con_nombre = ...   # TU CÓDIGO AQUÍ

print(len(viajes), '->', len(con_nombre))
con_nombre.head()

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(con_nombre) == len(viajes) - 1, 'inner debe botar exactamente el viaje huérfano'
assert 'nombre' in con_nombre.columns
assert 9 not in con_nombre['salida'].values
print('✅ ¡Correcto!')

### 1.2 left: nadie se pierde (pero aparece NaN) ⭐

Repite el merge con `how='left'` en `con_nombre_left` y calcula `huerfanos`: cuántos viajes quedaron **sin nombre** de estación (`isnull().sum()` sobre la columna `nombre`).

<details><summary>💡 Pista (haz clic)</summary>

inner responde «¿qué calza?»; left responde «¿qué le falta a mi tabla?» — los NaN que deja son un DIAGNÓSTICO, no un error.

</details>

In [ ]:
con_nombre_left = ...   # TU CÓDIGO AQUÍ
huerfanos = ...

print(len(con_nombre_left), 'viajes ·', huerfanos, 'sin estación conocida')

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert len(con_nombre_left) == len(viajes), 'left no debe perder ningún viaje'
assert huerfanos == 1
assert con_nombre_left.loc[con_nombre_left['salida'] == 9, 'nombre'].isnull().all()
print('✅ ¡Correcto!')

## 2 · Núcleo

### 2.1 Doble join: salida Y llegada ⭐ (ejercicio estrella) ⭐⭐

El reporte necesita el nombre de AMBAS estaciones. Haz dos merges encadenados sobre `viajes` usando `suffixes=('_salida', '_llegada')` en el segundo para distinguir las columnas repetidas. Guarda en `tabla` y deja solo `['id_viaje', 'nombre_salida', 'nombre_llegada', 'duracion_min']`.

<details><summary>💡 Pista (haz clic)</summary>

Sin `suffixes`, pandas inventa `_x` y `_y` — funciona, pero nadie entiende el reporte después. Nómbralos tú.

</details>

In [ ]:
paso1 = pd.merge(viajes, estaciones, left_on='salida',
                 right_on='id_estacion', how='left')
tabla = ...   # TU CÓDIGO AQUÍ: el segundo merge, por 'llegada', con suffixes

tabla

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert list(tabla.columns) == ['id_viaje', 'nombre_salida', 'nombre_llegada', 'duracion_min']
assert len(tabla) == len(viajes)
_v3 = tabla[tabla['id_viaje'] == 3].iloc[0]
assert _v3['nombre_salida'] == 'Plaza Perú' and _v3['nombre_llegada'] == 'Estación Central'
print('✅ ¡Correcto!')

### 2.2 Concat: llega el mes siguiente ⭐⭐

Llegaron los viajes de abril. Apílalos bajo los de marzo con `pd.concat([...], ignore_index=True)` en `anual`. Antes de eso, mira qué pasa SIN `ignore_index`: guarda en `repetidos` cuántas veces aparece la etiqueta 0 en el índice del concat ingenuo.

<details><summary>💡 Pista (haz clic)</summary>

concat apila conservando los índices originales: 0..11 y luego 0..5. `ignore_index=True` renumera de corrido.

</details>

In [ ]:
abril = pd.DataFrame({
    'id_viaje': range(13, 19),
    'salida': [2, 3, 1, 5, 4, 2],
    'llegada': [1, 1, 4, 2, 3, 5],
    'duracion_min': [10, 19, 21, 28, 13, 9],
})
ingenuo = ...     # TU CÓDIGO AQUÍ: concat SIN ignore_index
repetidos = ...   # cuántas filas responden a .loc[0]
anual = ...       # ahora con ignore_index=True

print('filas bajo la etiqueta 0:', repetidos)
print(len(anual), 'viajes en total')

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert repetidos == 2, 'sin ignore_index, la etiqueta 0 existe en marzo Y en abril'
assert len(anual) == len(viajes) + len(abril)
assert list(anual.index) == list(range(len(anual))), 'anual debe quedar renumerado 0..n-1'
print('✅ ¡Correcto!')

### 2.3 Validar el join como profesional ⭐⭐

Después de todo merge conviene auditar. Sobre `con_nombre_left` construye `auditoria`, un pequeño dict con: `'total'` (filas), `'con_nombre'` (filas con estación conocida) y `'sin_nombre'` (los NaN). Los tres números deben cuadrar.

In [ ]:
auditoria = {
    'total': ...,        # TU CÓDIGO AQUÍ
    'con_nombre': ...,
    'sin_nombre': ...,
}

print(auditoria)

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert auditoria['total'] == auditoria['con_nombre'] + auditoria['sin_nombre']
assert auditoria['total'] == len(viajes)
assert auditoria['sin_nombre'] == 1
print('✅ ¡Correcto!')

## 3 · Desafío

### 3.1 La estación fantasma 🏁 ⭐⭐⭐

Dos preguntas de auditoría, dos técnicas:
- `sin_uso`: el NOMBRE de la estación del catálogo que **nunca** aparece como salida (usa `~estaciones['id_estacion'].isin(...)`)
- `fuera_catalogo`: los id de salida que aparecen en viajes pero **no** existen en el catálogo (misma idea, invertida)

<details><summary>💡 Pista (haz clic)</summary>

`~mascara` invierte la máscara: «los que NO están en la lista». También sirve un merge outer con `indicator=True` — mira las categorías left_only/right_only que entrega.

</details>

In [ ]:
sin_uso = ...          # TU CÓDIGO AQUÍ
fuera_catalogo = ...

print('nunca usada como salida:', list(sin_uso))
print('salidas fuera de catálogo:', list(fuera_catalogo))

In [ ]:
# ✅ AUTOCHEQUEO — corre esta celda: si no reclama, vas bien
assert list(sin_uso) == ['Universidad']
assert list(fuera_catalogo) == [9]
print('✅ ¡Correcto!')

---

## 🎯 Chequea tu intuición

Antes de cerrar, abre el visualizador y en el botón **🎯 Ejercicios** corre una ronda de [Predice la salida](https://mati3939.github.io/visualizador-numpy-pandas/#predice) y [Detective de bugs](https://mati3939.github.io/visualizador-numpy-pandas/#detective) filtrando por **Joins**. Si un error de Python te deja pegado, pégalo en el [Traductor de errores](https://mati3939.github.io/visualizador-numpy-pandas/#traductor).